# Speculative decoding: English, Hindi and Urdu

Compare target-only and assisted generation with Qwen. Record throughput, target forward calls, exact output matches and script scores. Results describe the tested model pair and questions; saved numerical mismatches and quality limitations must be reported.


**Proposed validation run:** no saved Kaggle executions are supplied with this project. The 7B notebook uses two GPUs and is not a controlled hardware-only comparison.


## 0. Install
Kaggle already ships torch with CUDA — **do not reinstall it**. Only pin `transformers`, because this code depends on 4.57's speculative-decoding internals.

In [ ]:
!pip install -q "transformers==4.57.6" "accelerate>=1.0.0"

import transformers, torch
print("transformers", transformers.__version__)
print("torch       ", torch.__version__, "| CUDA", torch.version.cuda)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU accelerator!")

## 1. Settings

- `TARGET` big model (the checker), `DRAFT` small model (the helper)
- Both must share a tokenizer — that is why both are Qwen2.5
- `K_SWEEP` how many tokens the helper guesses per round
- Kept small (3 questions, 1 repeat, K in {2,4}) so this finishes in ~10 min.
  The laptop run did the full 10 x 3 x 5 x 2 = 300.

In [ ]:
import csv, json, random, time, subprocess
import statistics as st
from pathlib import Path

TARGET  = "Qwen/Qwen2.5-7B-Instruct"   # 3B on the laptop; bigger target -> smaller c
DRAFT   = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_NEW = 128
LANGS   = ["en", "hi", "ur"]
LIMIT   = 3           # questions per language (laptop run used all 10)
K_SWEEP = [1, 2, 4]      # laptop run used [2,4,6,8]
REPEATS = 3
SEED    = 0
REP_PENALTY = 1.05    # Qwen's own setting; 1.0 makes the model loop in Urdu

OUT = Path("/kaggle/working/results_new")
assert not (OUT / "raw.csv").exists(), "Choose a new OUT folder."
OUT.mkdir(parents=True, exist_ok=True)
LANG_NAMES = {"en": "English", "hi": "Hindi", "ur": "Urdu"}
SYSTEM = "Answer in the same language as the question."
print("ready")

## 2. The questions
Same 10 questions in each language, translated to match. Embedded here so nothing needs uploading.

In [ ]:
PROMPTS = {
    "en": [
        "Explain how rain forms.",
        "Write a short story about a boy and his kite.",
        "What are the benefits of drinking water?",
        "Explain in simple words what a computer does.",
        "Describe a typical morning in a busy city.",
        "Give some tips for preparing for an exam.",
        "Why is exercise important for health?",
        "Explain how a train works.",
        "Write a letter to a friend about your holidays.",
        "What is the importance of trees?"
    ],
    "hi": [
        "बारिश कैसे बनती है, समझाइए।",
        "एक लड़के और उसकी पतंग के बारे में एक छोटी कहानी लिखिए।",
        "पानी पीने के क्या फायदे हैं?",
        "आसान शब्दों में समझाइए कि कंप्यूटर क्या करता है।",
        "किसी व्यस्त शहर की एक आम सुबह का वर्णन कीजिए।",
        "परीक्षा की तैयारी के लिए कुछ सुझाव दीजिए।",
        "स्वास्थ्य के लिए व्यायाम क्यों ज़रूरी है?",
        "समझाइए कि रेलगाड़ी कैसे चलती है।",
        "अपनी छुट्टियों के बारे में एक दोस्त को पत्र लिखिए।",
        "पेड़ों का क्या महत्व है?"
    ],
    "ur": [
        "بارش کیسے بنتی ہے، سمجھائیے۔",
        "ایک لڑکے اور اس کی پتنگ کے بارے میں ایک مختصر کہانی لکھیے۔",
        "پانی پینے کے کیا فوائد ہیں؟",
        "آسان الفاظ میں سمجھائیے کہ کمپیوٹر کیا کرتا ہے۔",
        "کسی مصروف شہر کی ایک عام صبح کا حال بیان کیجیے۔",
        "امتحان کی تیاری کے لیے کچھ مشورے دیجیے۔",
        "صحت کے لیے ورزش کیوں ضروری ہے؟",
        "سمجھائیے کہ ریل گاڑی کیسے چلتی ہے۔",
        "اپنی چھٹیوں کے بارے میں ایک دوست کو خط لکھیے۔",
        "درختوں کی کیا اہمیت ہے؟"
    ]
}

for l in LANGS:
    print(l, len(PROMPTS[l]), "questions |", PROMPTS[l][0][:45])

## 3. Script screen

Count the fraction of alphabetic Unicode code points in the expected script. A 70% threshold screens script switching; it does not establish fluency, factuality or even language identity. New code checks each actual output, including divergent speculative answers.


In [ ]:
SCRIPTS = {
    "en": [(0x0041, 0x005A), (0x0061, 0x007A)],                      # Latin
    "hi": [(0x0900, 0x097F)],                                        # Devanagari
    "ur": [(0x0600, 0x06FF), (0x0750, 0x077F), (0xFB50, 0xFDFF), (0xFE70, 0xFEFF)],  # Arabic
}
LANG_OK_THRESHOLD = 0.7

def script_ratio(text, lang):
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    return sum(any(a <= ord(c) <= b for a, b in SCRIPTS[lang]) for c in letters) / len(letters)

print(script_ratio("Rain forms", "en"), script_ratio("बारिश बनती है", "hi"), script_ratio("Rain forms", "ur"))

## 4. Measuring instruments

- **PassCounter** hooks the big model and counts how many times it actually runs.
  `tau = tokens produced / big-model runs`. Normal decoding = 1.0. Higher = guesses landing.
- **gpu_clock_mhz** records the GPU clock per run, sampled *outside* the timed region.
  On the laptop the clock silently halved mid-run once and quietly ruined the timings.

In [ ]:
class PassCounter:
    def __init__(self, model):
        self.n = 0
        model.register_forward_hook(self._hook)
    def _hook(self, *_):
        self.n += 1

def gpu_clock_mhz():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=clocks.sm", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=5).stdout.strip().splitlines()[0]
        return int(out)
    except Exception:
        return None

print("GPU clock now:", gpu_clock_mhz(), "MHz")

## 5. Load both models
`float16` halves memory. Kaggle's 16 GB is roomy — the laptop had 8 GB and was at 85%.

**Run this cell once.** Re-running loads a second copy; restart the session instead.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "No GPU - Settings > Accelerator > GPU T4 x2"
N_GPU = torch.cuda.device_count()
GPU   = torch.cuda.get_device_name(0)
print(f"{N_GPU} GPU(s): {GPU}")

# 7B + 0.5B = 15.11 GiB and a T4 only has ~14.6 GiB free, so they cannot share
# one card. Target on GPU 0, draft on GPU 1. Each model stays whole on its own
# card, so there is no layer-split overhead inside a forward pass.
TARGET_DEV = "cuda:0"
DRAFT_DEV  = "cuda:1" if N_GPU > 1 else "cuda:0"
assert N_GPU >= 2, "7B FP16 setup requires two T4 GPUs; use the 3B notebook for one GPU."
assert N_GPU >= 2, "7B FP16 setup requires two T4 GPUs; use the 3B notebook for one GPU."
assert N_GPU >= 2, "7B FP16 setup requires two T4 GPUs; use the 3B notebook for one GPU."
if N_GPU < 2:
    print("WARNING: only 1 GPU. A 7B target + 0.5B draft will OOM on a single T4.")
    print("         Set Accelerator to 'GPU T4 x2', or use TARGET = Qwen/Qwen2.5-3B-Instruct.")

def load(name, dev):
    m = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16, device_map={"": dev})
    m.eval(); return m

tok    = AutoTokenizer.from_pretrained(TARGET)
target = load(TARGET, TARGET_DEV)   # ~15 GB download on first run
draft  = load(DRAFT,  DRAFT_DEV)    # ~1 GB
counter = PassCounter(target)

for i in range(N_GPU):
    used  = torch.cuda.memory_allocated(i) / 2**30
    total = torch.cuda.get_device_properties(i).total_memory / 2**30
    print(f"  cuda:{i} {torch.cuda.get_device_name(i)}: {used:5.2f} / {total:5.2f} GiB")
print(f"target on {TARGET_DEV} | draft on {DRAFT_DEV}")

## 6. Pinning the draft strategy

In Transformers 4.57.6, default K=20 is constant with a confidence gate. With scikit-learn, confidence state can be written back across calls. Before every assisted call, pin K, the constant schedule and confidence threshold 0, disabling that adaptive behavior. K remains a maximum: EOS and the final generation budget may shorten a round.


In [ ]:
def set_draft_strategy(k):
    # Must run before EVERY assisted generate() - transformers mutates this in place
    gc = draft.generation_config
    gc.num_assistant_tokens = int(k)
    gc.num_assistant_tokens_schedule = "constant"
    gc.assistant_confidence_threshold = 0.0

print("HF defaults ->", draft.generation_config.num_assistant_tokens,
      draft.generation_config.num_assistant_tokens_schedule,
      draft.generation_config.assistant_confidence_threshold)
set_draft_strategy(2)
print("pinned      ->", draft.generation_config.num_assistant_tokens,
      draft.generation_config.assistant_confidence_threshold)

## 7. Core functions

Encode the question with Qwen's chat template. Time synchronized target.generate, with assistant_model=draft for speculation. Set repetition_penalty explicitly; HF reuses target settings/processors for the assistant. do_sample=False selects the largest processed score, not necessarily the largest raw model logit.


In [ ]:
def synchronize_all():
    for device in range(torch.cuda.device_count()):
        torch.cuda.synchronize(device)

GREEDY = dict(do_sample=False, temperature=None, top_p=None, top_k=None)

def encode(prompt):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return tok(text, return_tensors="pt").to(TARGET_DEV)

@torch.inference_mode()
def generate(inputs, max_new, k=0):
    kwargs = dict(max_new_tokens=max_new, pad_token_id=tok.eos_token_id,
                  repetition_penalty=REP_PENALTY, **GREEDY)
    if k:
        set_draft_strategy(k)
        kwargs["assistant_model"] = draft      # <-- speculative decoding ON
    counter.n = 0
    synchronize_all(); t0 = time.perf_counter()
    out = target.generate(**inputs, **kwargs)
    synchronize_all(); dt = time.perf_counter() - t0
    return out[0, inputs["input_ids"].shape[1]:].tolist(), dt, counter.n

warm = encode("Hello")
generate(warm, 16, 0); generate(warm, 16, K_SWEEP[0])
print("warmed up")

## Calibration: generation-cost proxy

Measure synchronized wall time per generated token, including prefill and bookkeeping, with both models resident. Use three interleaved repeats. This is a cost proxy; a profiler is needed to identify overhead sources. `tau / (1 + K*c)` is an approximation, not a guaranteed bound for the measured implementation.


In [ ]:
@torch.inference_mode()
def pass_costs(n=64):
    samples = {"target": [], "draft": []}
    models = (("target", target), ("draft", draft))
    def run(model, count):
        inputs = {k: v.to(model.device) for k, v in encode("Explain how rain forms.").items()}
        synchronize_all()
        t0 = time.perf_counter()
        model.generate(**inputs, max_new_tokens=count, min_new_tokens=count,
                       pad_token_id=tok.eos_token_id, repetition_penalty=REP_PENALTY, **GREEDY)
        synchronize_all()
        return (time.perf_counter() - t0) / count
    for name, model in models:
        run(model, 8); run(model, 8)
    for rep in range(3):
        for name, model in (models if rep % 2 == 0 else models[::-1]):
            samples[name].append(run(model, n))
    return st.median(samples["target"]), st.median(samples["draft"]), samples

T_PASS, D_PASS, CALIBRATION_SAMPLES = pass_costs()
C = D_PASS / T_PASS
counter.n = 0
print(f"{TARGET}: {T_PASS*1e3:.2f} ms/token proxy; draft {D_PASS*1e3:.2f}; c={C:.3f}")
print("Includes prefill/bookkeeping; cross-device copies also matter in a two-GPU sweep.")


## 9. One question, both ways
See the mechanism before the full loop. Watch the pass count drop while the clock time doesn't.

In [ ]:
q = encode("Explain how rain forms.")
a, at, ap = generate(q, MAX_NEW, 0)
b, bt, bp = generate(q, MAX_NEW, 4)
print(f"normal:      {len(a)} tokens, {at:.2f}s, {ap} big-model passes, tau {len(a)/ap:.2f}")
print(f"speculative: {len(b)} tokens, {bt:.2f}s, {bp} big-model passes, tau {len(b)/max(bp,1):.2f}  (K=4)")
print(f"speedup {(len(b)/bt)/(len(a)/at):.2f}x | identical: {a == b}")
print("\n", tok.decode(a, skip_special_tokens=True)[:300])

## Shuffled sweep

Build baseline and speculative conditions for every question, shuffle each repeat, and record time, tokens and target calls. Shuffling spreads drift; it does not eliminate it. Medians of two trials equal their mean. Exact-match checks run after baselines are available. Saved IDs/text allow later inspection of divergences.


In [ ]:
conditions = [(l, i, k) for l in LANGS for i in range(len(PROMPTS[l][:LIMIT])) for k in [0] + K_SWEEP]
rows, canonical, ids_by_row = [], {}, []
rng = random.Random(SEED); done = 0; t0 = time.perf_counter()

for rep in range(REPEATS):
    order = conditions[:]; rng.shuffle(order)
    for lang, i, k in order:
        ids, dt, passes = generate(encode(PROMPTS[lang][i]), MAX_NEW, k)
        if k == 0 and (lang, i) not in canonical:
            canonical[(lang, i)] = ids
        done += 1; tau = len(ids) / max(passes, 1); ids_by_row.append(ids)
        rows.append(dict(repeat=rep, lang=lang, prompt_id=i, k=k,
                         mode="baseline" if k == 0 else f"spec_k{k}",
                         tokens=len(ids), sec=dt, tps=len(ids)/dt,
                         passes=passes, tau=tau, matches_baseline=None,
                         sm_mhz=gpu_clock_mhz(), wall=round(time.perf_counter()-t0, 1)))
        print(f"[{done:3d}/{len(conditions)*REPEATS}] {lang} p{i} "
              f"{('baseline' if k==0 else 'K='+str(k)):>9s} | {len(ids)/dt:6.1f} tok/s | "
              f"{passes:4d} passes | tau {tau:.2f}")

# resolved after the loop: the order is shuffled, so a speculative run can land
# before that question's first baseline
for r, ids in zip(rows, ids_by_row):
    r["matches_baseline"] = ids == canonical[(r["lang"], r["prompt_id"])]
print(f"\ndone in {(time.perf_counter()-t0)/60:.1f} min")

## Summary

Screen every actual output by Unicode script; this does not assess language quality. Use median throughput within a prompt and then median prompt speedup ratios. Tau includes the target bonus/correction token. With two repeats the median equals the mean. SD across prompt ratios is variation across questions, not a confidence interval.


In [ ]:
def text_metrics(tok, ids, lang):
    text = tok.decode(ids, skip_special_tokens=True)
    sr = script_ratio(text, lang)
    return text, dict(lang_ok=sr >= LANG_OK_THRESHOLD, script_ratio=sr,
                      tokens_per_word=len(ids) / max(len(text.split()), 1),
                      chars_per_token=len(text) / max(len(ids), 1),
                      language_screen="unicode_script_only")

def finalise_rows(rows, ids_by_row, canonical, tok):
    """Screen each actual output and retain evidence for every divergence."""
    evidence = []
    for r, ids in zip(rows, ids_by_row):
        reference = canonical[(r["lang"], r["prompt_id"])]
        r["matches_baseline"] = ids == reference
        text, meta = text_metrics(tok, ids, r["lang"])
        r.update(meta)
        first = next((i for i, (a, b) in enumerate(zip(ids, reference)) if a != b), None)
        if first is None and len(ids) != len(reference):
            first = min(len(ids), len(reference))
        evidence.append(dict(repeat=r["repeat"], lang=r["lang"], prompt_id=r["prompt_id"],
                             k=r["k"], token_ids=ids, output=text,
                             matches_baseline=r["matches_baseline"],
                             first_divergence_token_0based=first, **meta))
    return evidence

def med(xs):
    return round(st.median(xs), 3)

def summarise(rows, args, gpu, t_pass, d_pass, c):
    """Median over repeats per (lang, prompt, k), then median over prompts."""
    ok = [r for r in rows if r["lang_ok"]]
    summary = dict(
        gpu=gpu, target=args.target, draft=args.draft, max_new=args.max_new,
        repeats=args.repeats, seed=args.seed, k_sweep=args.k,
        rep_penalty=args.rep_penalty,
        target_ms_per_pass=round(t_pass * 1e3, 2),
        draft_ms_per_pass=round(d_pass * 1e3, 2),
        c=round(c, 3),
        langs={},
    )
    for lang in args.langs:
        lr = [r for r in ok if r["lang"] == lang]
        all_lang = [r for r in rows if r["lang"] == lang]
        if not lr or not all_lang:
            continue
        pids = sorted({r["prompt_id"] for r in lr})
        base = {}
        for p in pids:
            b = [r["tps"] for r in lr if r["prompt_id"] == p and r["k"] == 0]
            if b:
                base[p] = st.median(b)
        if not base:
            continue
        drift = [r["tps"] for r in all_lang if r["k"] == 0]
        clocks = [r["sm_mhz"] for r in all_lang if r.get("sm_mhz")]
        entry = dict(
            n_prompts=len(base),
            sm_mhz_min=min(clocks) if clocks else None,
            sm_mhz_max=max(clocks) if clocks else None,
            sm_mhz_median=round(st.median(clocks)) if clocks else None,
            lang_ok_rate=round(len(lr) / len(all_lang), 3),
            base_tps=round(st.median(base.values()), 2),
            baseline_tps_min=round(min(drift), 2),
            baseline_tps_max=round(max(drift), 2),
            tokens_per_word=med([r["tokens_per_word"] for r in lr if r["k"] == 0]),
            chars_per_token=med([r["chars_per_token"] for r in lr if r["k"] == 0]),
            by_k={},
        )
        for k in args.k:
            kr = [r for r in lr if r["k"] == k and r["prompt_id"] in base]
            if not kr:
                continue
            speedups, taus = [], []
            for p in base:
                pr = [r for r in kr if r["prompt_id"] == p]
                if not pr:
                    continue
                speedups.append(st.median([r["tps"] for r in pr]) / base[p])
                taus.append(st.median([r["tau"] for r in pr]))
            if not speedups:
                continue
            tau_pooled = sum(r["tokens"] for r in kr) / max(sum(r["passes"] for r in kr), 1)
            entry["by_k"][str(k)] = dict(
                spec_tps=round(st.median(r["tps"] for r in kr), 2),
                speedup=med(speedups),
                speedup_mean=round(st.mean(speedups), 3),
                speedup_sd=round(st.pstdev(speedups), 3) if len(speedups) > 1 else 0.0,
                tau=med(taus),
                tau_pooled=round(tau_pooled, 3),
                predicted_speedup=round(tau_pooled / (1 + k * c), 3),
                identical_rate=round(sum(bool(r["matches_baseline"]) for r in kr) / len(kr), 2),
                n_runs=len(kr), n_prompts=len(speedups),
            )
        if entry["by_k"]:
            best = max(entry["by_k"], key=lambda k: entry["by_k"][k]["speedup"])
            entry["best_k"] = int(best)
            entry["best_speedup"] = entry["by_k"][best]["speedup"]
        summary["langs"][lang] = entry
    return summary

evidence = finalise_rows(rows, ids_by_row, canonical, tok)
(OUT / "run_outputs.json").write_text(json.dumps(evidence, ensure_ascii=False, indent=2), encoding="utf-8")
samples = []
for (lang, i), ids in canonical.items():
    answer, meta = text_metrics(tok, ids, lang)
    samples.append(dict(lang=lang, prompt_id=i, prompt=PROMPTS[lang][i], output=answer, **meta))
print("Exact speculative matches:", sum(r["matches_baseline"] for r in rows if r["k"]),
      "/", sum(bool(r["k"]) for r in rows))

from types import SimpleNamespace
ARGS = SimpleNamespace(target=TARGET, draft=DRAFT, max_new=MAX_NEW,
                       repeats=REPEATS, seed=SEED, k=K_SWEEP,
                       rep_penalty=REP_PENALTY, langs=LANGS)
summary = summarise(rows, ARGS, GPU, T_PASS, D_PASS, C)
for lang, entry in summary["langs"].items():
    print(lang, entry["by_k"])


## 12. Chart + save
Files land in `/kaggle/working/` — download them from the right-hand **Output** panel.

In [ ]:
import matplotlib.pyplot as plt

colors = {"en": "#4C6EF5", "hi": "#F59F00", "ur": "#E8590C"}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), dpi=150)
for l, v in summary["langs"].items():
    ks = sorted(int(k) for k in v["by_k"])
    ax1.plot(ks, [v["by_k"][str(k)]["speedup"] for k in ks], "o-", color=colors[l], label=LANG_NAMES[l])
    ax2.plot(ks, [v["by_k"][str(k)]["tau_pooled"] for k in ks], "o-", color=colors[l], label=LANG_NAMES[l])
ax1.axhline(1.0, color="#868E96", lw=1, ls="--")
ax1.set_xlabel("K"); ax1.set_ylabel("Speedup vs normal"); ax1.set_title("Speedup", fontsize=10)
ax2.set_xlabel("K"); ax2.set_ylabel("tau"); ax2.set_title("Tokens per target pass (includes bonus)", fontsize=10)
for ax in (ax1, ax2):
    ax.legend(fontsize=8, frameon=False)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
fig.suptitle(f"{DRAFT.split('/')[-1]} -> {TARGET.split('/')[-1]} on {GPU}  (c = {C:.2f})", fontsize=9)
fig.tight_layout(); fig.savefig(OUT / "sweep_kaggle.png"); plt.show()

with open(OUT / "raw.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
(OUT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
(OUT / "outputs.json").write_text(json.dumps(samples, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved to", OUT, "->", [p.name for p in OUT.iterdir()])
import platform, transformers
manifest = dict(target=TARGET, draft=DRAFT, gpu=GPU, max_new=MAX_NEW,
                repeats=REPEATS, k_sweep=K_SWEEP, seed=SEED, rep_penalty=REP_PENALTY,
                python=platform.python_version(), torch=torch.__version__,
                transformers=transformers.__version__, cuda=torch.version.cuda,
                target_device=str(target.device), draft_device=str(draft.device),
                target_revision=getattr(target.config, "_commit_hash", None),
                draft_revision=getattr(draft.config, "_commit_hash", None),
                calibration_seconds_per_token=CALIBRATION_SAMPLES,
                calibration="three interleaved end-to-end measurements; includes prefill",
                target_generation_config=target.generation_config.to_dict(),
                draft_generation_config=draft.generation_config.to_dict())
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")


## What to take back to the slides

Report measured throughput and tokens per target pass with the model pair, device placement and prompt count. Save all token IDs and match rates. A matching cost estimate does not prove an overhead cause. These notebooks are proposed checks; no Kaggle results are saved in the local project.
